# Versionnage et traçage des données

Nous allons étudier dans ces travaux pratiques comment versionner ses données et conserver des informations d'origine à l'aide de l'outil DVC.

Cet outil découple le versionnage des données et leur stockage.

## Configuration & installations — À exécuter

In [ ]:
!rm -rf sample_data .config
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!apt install tree
!pip install dvc dvc-s3

## Création du dépôt git

Créez un compte sur [DagsHub](https://dagshub.com/), puis créez un dépôt git vierge.

Une fois le dépôt créé, cliquez sur « *Get started with Data* » et récupérez votre token secret dans la partie « *Connection credentials* ».

Remplissez les variables `owner`, `repo` & `token` avec respectivement votre nom d'utilisateur DagsHub, votre nom de dépôt git et votre token secret.

In [ ]:
owner = ""
repo = ""
token = ""

Exécutez maintenant la cellule ci-dessous pour mettre en place votre environnement.

In [ ]:
!git init
!git remote add origin https://{token}@dagshub.com/{owner}/{repo}.git

## Initialisation du dépôt DVC

In [ ]:
!dvc init

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

### Solution

In [ ]:
!dvc init

In [ ]:
!git commit -m "Initialisation de DVC"

In [ ]:
!git push origin main

## Récupération d'une première version des données

Il y a plusieurs manières d'importer une source de données dans DVC. L'une d'entre elles est d'utiliser deux commandes&nbsp;: [`dvc get`](https://dvc.org/doc/command-reference/get) d'abord, qui récupère des données depuis un dépôt git/DVC, puis [`dvc add`](https://dvc.org/doc/command-reference/add) qui ajoute les données aux données gérées par DVC.

Utilisez ces deux commandes pour importer le fichier `wiki_movie_plots_deduped.csv` du dépôt git `https://github.com/m09/dataset-wikipedia-movie-plots/` sous le nom `data.csv`.

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

DVC utilise un dossier de cache qui contient (la plupart du temps) les données utilisées par le projet. Vous pouvez observer ce dossier à l'aide de la commande `!tree .dvc`. Que constatez-vous ?

N'hésitez pas à observer les fichiers d'extension `.dvc` avec la commande `!cat nom-du-fichier.dvc`.

### Solution

In [ ]:
!dvc get https://github.com/m09/dataset-wikipedia-movie-plots wiki_movie_plots_deduped.csv -o data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc

In [ ]:
!tree .dvc

## Ajout du fichier à un commit git

Commitez maintenant ce fichier en suivant les recommandations de DVC données en sortie de la commande `dvc add`.

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

### Solution

In [ ]:
!git add .gitignore data.csv.dvc

In [ ]:
!git commit -m "Ajout d'une première version des données"

In [ ]:
!git tag "v1"

In [ ]:
!git push origin main v1

## Configuration d'un serveur de stockage

DVC peut stocker les données dans divers types de serveurs. DagsHub met à disposition un espace de stockage qui s'utilise comme un bucket s3 (solution de stockage du cloud amazon).

Configurez ce serveur en suivant les instructions disponibles dans l'onglet « *Data* » du bouton vert « *Remote* » de votre dépôt DagsHub puis publiez-y vos données.

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

### Solution

In [ ]:
!dvc remote add origin s3://dvc
!dvc remote modify origin endpointurl https://dagshub.com/{owner}/{repo}.s3
!dvc remote modify origin --local access_key_id {token}
!dvc remote modify origin --local secret_access_key {token}

In [ ]:
!dvc push -r origin

## Modification des données

Exécutez les cellules suivantes pour modifier le fichier de données&nbsp;:

In [ ]:
!sort -r < data.csv > a && dvc remove data.csv.dvc && mv a data.csv

Utilisez maintenant `dvc add` & `git add` pour enregistrer ce changement.

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

### Solution

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc

In [ ]:
!git add data.csv.dvc .gitignore

In [ ]:
!git commit -m "Données v2"
!git tag "v2"

In [ ]:
!git push origin main v2

In [ ]:
!dvc push -r origin

## Retour aux données originales

On imagine avoir détecté un problème avec nos nouvelles données&nbsp;: on souhaite revenir à la première version. Utilisez `git checkout` pour revenir à la version antérieure des métadonnées, puis `dvc checkout` pour récupérer le fichier correspondant.

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

Commitez maintenant ce retour à la première version.

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

### Solution

In [ ]:
!git checkout v1 data.csv.dvc

In [ ]:
!cat data.csv.dvc
!tree .dvc

In [ ]:
!dvc checkout data.csv

In [ ]:
!git add data.csv.dvc

In [ ]:
!git commit -m "Retour aux données v1"
!git tag v3

In [ ]:
!git push origin main v3

In [ ]:
!dvc push -r origin

## Traitement des données

Exécutez la cellule suivante qui contient un script Python qui prend deux arguments, et qui écrit au chemin donné par le deuxième argument le contenu du fichier au chemin donné par le premier argument, en passant le contenu en majuscule.

Cette cellule va écrire ce script au chemin `upper.py`

In [ ]:
%%writefile upper.py
from pathlib import Path
from sys import argv

Path(argv[2]).write_text(
    Path(argv[1]).read_text(encoding="utf8").upper(),
    encoding="utf8")

Ajoutez une étape de traitement avec [`dvc stage add`](https://dvc.org/doc/command-reference/stage/add) qui prend en entrée le fichier `data.csv` et produit le fichier `data-upper.csv` à partir de ce script.

In [ ]:
!# Votre commande ici, notez bien le ! qui préfixe les commandes bash dans Colab

Exécutez maintenant cette étape de traitement avec [`dvc repro`](https://dvc.org/doc/command-reference/repro) puis sauvegardez les données dans le serveur de stockage et les métadonnées dans git.

### Solution

In [ ]:
!dvc stage add -n transform-uppercase -d data.csv -o data-upper.csv python upper.py data.csv data-upper.csv

In [ ]:
!dvc repro

In [ ]:
!git add dvc.yaml .gitignore dvc.lock

In [ ]:
!git commit -m "Pipeline pour passer un fichier en majuscule"

In [ ]:
!git push origin main

In [ ]:
!dvc push -r origin